In [1]:
import numpy as np
import pandas as pd

from scipy.sparse import csr_matrix, hstack
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.model_selection import TimeSeriesSplit
from sklearn.metrics import (
    mean_absolute_error,
    mean_squared_error,
    median_absolute_error,
    r2_score
)

import lightgbm as lgb

In [2]:
df = pd.read_csv("steam_games_cleaned.csv")

df["release_date"] = pd.to_datetime(df["release_date"])
df = df.sort_values("release_date").reset_index(drop=True)

In [3]:
df["log_avg_playtime"] = np.log1p(df["avg_playtime"])

df["release_year"] = df["release_date"].dt.year
df["release_month"] = df["release_date"].dt.month

In [4]:
# --------------------------------------------------
# counts
# --------------------------------------------------

df["developer_games_before"] = (
    df.groupby("developers")
      .cumcount()
      .fillna(0)
)

df["publisher_games_before"] = (
    df.groupby("publishers")
      .cumcount()
      .fillna(0)
)

df["developer_is_new"] = (
    df["developer_games_before"] == 0
).astype(int)

df["publisher_is_new"] = (
    df["publisher_games_before"] == 0
).astype(int)

df["developer_experience"] = np.log1p(
    df["developer_games_before"]
)

df["publisher_experience"] = np.log1p(
    df["publisher_games_before"]
)

In [5]:
df["developer_price_std_before"] = (
    df.groupby("developers")["price_usd"]
      .transform(
          lambda x:
          x.expanding()
           .std()
           .shift(1)
      )
).fillna(0)

df["publisher_price_std_before"] = (
    df.groupby("publishers")["price_usd"]
      .transform(
          lambda x:
          x.expanding()
           .std()
           .shift(1)
      )
).fillna(0)

In [6]:
global_rating = df["rating"].mean()

dev_rating_sum = (
    df.groupby("developers")["rating"].cumsum()
    - df["rating"]
)

pub_rating_sum = (
    df.groupby("publishers")["rating"].cumsum()
    - df["rating"]
)

dev_count = df["developer_games_before"].replace(
    0, np.nan
)

pub_count = df["publisher_games_before"].replace(
    0, np.nan
)

df["developer_mean_rating_before"] = (
    dev_rating_sum / dev_count
).fillna(global_rating)

df["publisher_mean_rating_before"] = (
    pub_rating_sum / pub_count
).fillna(global_rating)

df["developer_total_reviews_before"] = (
    df.groupby("developers")["total_reviews"]
      .cumsum()
      - df["total_reviews"]
).fillna(0)

df["publisher_total_reviews_before"] = (
    df.groupby("publishers")["total_reviews"]
      .cumsum()
      - df["total_reviews"]
).fillna(0)

In [7]:
texts = df["tags_string"].fillna("")

In [8]:
num_features = [
    # CORE
    "tags_count_actual",
    "release_year",
    "release_month",
    "log_avg_playtime",

    # DEV
    "developer_games_before",
    "developer_experience",
    "developer_is_new",
    "developer_price_std_before",
    "developer_mean_rating_before",
    "developer_total_reviews_before",

    # PUBLISHER
    "publisher_games_before",
    "publisher_experience",
    "publisher_is_new",
    "publisher_price_std_before",
    "publisher_mean_rating_before",
    "publisher_total_reviews_before",
]

In [9]:
# ============================================================
# 3. PRICE FILTER
# ============================================================

df = df[
    (df["price_usd"] >= 4.99) &
    (df["price_usd"] <= 29.99)
].copy()

print(df.shape)

(30856, 31)


In [10]:
X_num = df[num_features]
y = df["price_usd"]

In [11]:
tscv = TimeSeriesSplit(n_splits=10)

In [12]:
def evaluate(y_true, y_pred):
    return {
        "MAE": mean_absolute_error(
            y_true, y_pred
        ),
        "MSE": mean_squared_error(
            y_true, y_pred
        ),
        "MedAE": median_absolute_error(
            y_true, y_pred
        ),
        "R2": r2_score(
            y_true, y_pred
        ),
        "MAPE": np.mean(
            np.abs(
                (y_true - y_pred)
                / y_true
            )
        ) * 100,
        "SMAPE": np.mean(
            2*np.abs(y_pred-y_true)
            /
            (np.abs(y_true)+np.abs(y_pred))
        ) * 100,
        "WAPE": (
            np.sum(
                np.abs(y_true-y_pred)
            )
            /
            np.sum(y_true)
        ) * 100,
        "Bias": np.mean(
            y_pred-y_true
        )
    }

In [13]:
# Инициализация
results = []
# Создаём списки, но сохраняем индексы
oof_true_list = []
oof_pred_list = []
oof_idx_list = []  # КЛЮЧЕВОЙ МОМЕНТ: сохраняем индексы

feature_gain_folds = []
perm_folds = []
shap_folds = []

N_PERM = 1000
N_SHAP = 500

# TF-IDF векторизатор (создаём один раз)
tfidf = TfidfVectorizer(max_features=5000, stop_words='english')
texts = df["tags_string"].fillna("")

In [14]:
for fold, (train_idx, val_idx) in enumerate(tscv.split(X_num)):
    print(f"\nFOLD {fold + 1}")
    
    # === 1. Разделение данных ===
    X_num_train, X_num_val = X_num.iloc[train_idx], X_num.iloc[val_idx]
    y_train, y_val = y.iloc[train_idx], y.iloc[val_idx]
    
    # === 2. TF-IDF для текстов ===
    # ВАЖНО: fit только на train, transform на train и val
    tfidf.fit(texts.iloc[train_idx])
    X_text_train = tfidf.transform(texts.iloc[train_idx])
    X_text_val = tfidf.transform(texts.iloc[val_idx])
    
    # === 3. Объединение числовых и текстовых признаков ===
    X_train = hstack([X_num_train, X_text_train])
    X_val = hstack([X_num_val, X_text_val])
    
    # === 4. Обучение модели ===
    model = lgb.LGBMRegressor(
        n_estimators=300,
        learning_rate=0.05,
        max_depth=5,
        random_state=42,
        verbose=-1
    )
    
    model.fit(
        X_train, y_train,
        eval_set=[(X_val, y_val)],
        eval_metric='mae',
        callbacks=[lgb.early_stopping(20), lgb.log_evaluation(50)]
    )
    
    # === 5. Предсказание ===
    y_pred = model.predict(X_val)
    
    # === 6. Сохранение OOF по индексам (КЛЮЧЕВОЕ ИСПРАВЛЕНИЕ) ===
    oof_true_list.append(y_val.values)
    oof_pred_list.append(y_pred)
    oof_idx_list.append(val_idx)
    
    # === 7. Метрики для фолда ===
    fold_results = evaluate(y_val, y_pred)
    fold_results['fold'] = fold + 1
    results.append(fold_results)


FOLD 1
Training until validation scores don't improve for 20 rounds
[50]	valid_0's l1: 3.95536	valid_0's l2: 27.6606
[100]	valid_0's l1: 3.92036	valid_0's l2: 27.2966
Early stopping, best iteration is:
[103]	valid_0's l1: 3.91889	valid_0's l2: 27.2823

FOLD 2


c:\Users\alex\AppData\Local\Programs\Python\Python312\Lib\site-packages\sklearn\utils\validation.py:2691: UserWarning: X does not have valid feature names, but LGBMRegressor was fitted with feature names
  warnings.warn(
c:\Users\alex\AppData\Local\Programs\Python\Python312\Lib\site-packages\lightgbm\basic.py:1238: UserWarning: Converting data to scipy sparse matrix.
  _log_warning("Converting data to scipy sparse matrix.")


Training until validation scores don't improve for 20 rounds
[50]	valid_0's l1: 4.30438	valid_0's l2: 33.0653
[100]	valid_0's l1: 4.24654	valid_0's l2: 32.4039
[150]	valid_0's l1: 4.24217	valid_0's l2: 32.2393
Early stopping, best iteration is:
[135]	valid_0's l1: 4.24041	valid_0's l2: 32.2662

FOLD 3


c:\Users\alex\AppData\Local\Programs\Python\Python312\Lib\site-packages\sklearn\utils\validation.py:2691: UserWarning: X does not have valid feature names, but LGBMRegressor was fitted with feature names
  warnings.warn(
c:\Users\alex\AppData\Local\Programs\Python\Python312\Lib\site-packages\lightgbm\basic.py:1238: UserWarning: Converting data to scipy sparse matrix.
  _log_warning("Converting data to scipy sparse matrix.")


Training until validation scores don't improve for 20 rounds
[50]	valid_0's l1: 4.17895	valid_0's l2: 30.3084
[100]	valid_0's l1: 4.12004	valid_0's l2: 29.2089
[150]	valid_0's l1: 4.09539	valid_0's l2: 28.8535
[200]	valid_0's l1: 4.08308	valid_0's l2: 28.7017
[250]	valid_0's l1: 4.0701	valid_0's l2: 28.5264
[300]	valid_0's l1: 4.06429	valid_0's l2: 28.4436
Did not meet early stopping. Best iteration is:
[282]	valid_0's l1: 4.06369	valid_0's l2: 28.4423

FOLD 4


...

c:\Users\alex\AppData\Local\Programs\Python\Python312\Lib\site-packages\sklearn\utils\validation.py:2691: UserWarning: X does not have valid feature names, but LGBMRegressor was fitted with feature names
  warnings.warn(
c:\Users\alex\AppData\Local\Programs\Python\Python312\Lib\site-packages\lightgbm\basic.py:1238: UserWarning: Converting data to scipy sparse matrix.
  _log_warning("Converting data to scipy sparse matrix.")



FOLD 9
Training until validation scores don't improve for 20 rounds
[50]	valid_0's l1: 4.27523	valid_0's l2: 29.4879
[100]	valid_0's l1: 4.14567	valid_0's l2: 28.1515
[150]	valid_0's l1: 4.10793	valid_0's l2: 27.7672
[200]	valid_0's l1: 4.08464	valid_0's l2: 27.5234
[250]	valid_0's l1: 4.06886	valid_0's l2: 27.4087
[300]	valid_0's l1: 4.05335	valid_0's l2: 27.3166
Did not meet early stopping. Best iteration is:
[300]	valid_0's l1: 4.05335	valid_0's l2: 27.3166

FOLD 10


c:\Users\alex\AppData\Local\Programs\Python\Python312\Lib\site-packages\sklearn\utils\validation.py:2691: UserWarning: X does not have valid feature names, but LGBMRegressor was fitted with feature names
  warnings.warn(
c:\Users\alex\AppData\Local\Programs\Python\Python312\Lib\site-packages\lightgbm\basic.py:1238: UserWarning: Converting data to scipy sparse matrix.
  _log_warning("Converting data to scipy sparse matrix.")


Training until validation scores don't improve for 20 rounds
[50]	valid_0's l1: 4.27733	valid_0's l2: 29.5408
[100]	valid_0's l1: 4.12785	valid_0's l2: 28.0293
[150]	valid_0's l1: 4.0801	valid_0's l2: 27.567
[200]	valid_0's l1: 4.05123	valid_0's l2: 27.3164
[250]	valid_0's l1: 4.02927	valid_0's l2: 27.1162
[300]	valid_0's l1: 4.01073	valid_0's l2: 26.9454
Did not meet early stopping. Best iteration is:
[300]	valid_0's l1: 4.01073	valid_0's l2: 26.9454


c:\Users\alex\AppData\Local\Programs\Python\Python312\Lib\site-packages\sklearn\utils\validation.py:2691: UserWarning: X does not have valid feature names, but LGBMRegressor was fitted with feature names
  warnings.warn(
c:\Users\alex\AppData\Local\Programs\Python\Python312\Lib\site-packages\lightgbm\basic.py:1238: UserWarning: Converting data to scipy sparse matrix.
  _log_warning("Converting data to scipy sparse matrix.")


In [15]:
# === После цикла ===
print("\n" + "="*50)
print("Итоговые метрики:")
print("="*50)

results_df_folds = pd.DataFrame(results)
print(results_df_folds.mean())


Итоговые метрики:
MAE       4.110338
MSE      28.614600
MedAE     3.353475
R2        0.298284
MAPE     43.413552
SMAPE    36.649185
WAPE     36.058059
Bias     -0.181577
fold      5.500000
dtype: float64


In [16]:
oof_true = np.concatenate(oof_true_list)
oof_pred = np.concatenate(oof_pred_list)
indices_all = np.concatenate(oof_idx_list)

# ВАЖНО: создаём DataFrame с ПРАВИЛЬНЫМИ индексами
cv_results = pd.DataFrame({
    'true_price': oof_true,
    'pred_price': oof_pred
}, index=indices_all)  # ← ИСПОЛЬЗУЕМ СОХРАНЁННЫЕ ИНДЕКСЫ

# Теперь join будет работать корректно, потому что индексы совпадают
cv_results = cv_results.join(df[['name', 'release_date', 'price_usd']])

In [17]:
res_df = pd.DataFrame(results)

print("\nCV MEAN")
print(res_df.mean())

print("\nCV STD")
print(res_df.std())


CV MEAN
MAE       4.110338
MSE      28.614600
MedAE     3.353475
R2        0.298284
MAPE     43.413552
SMAPE    36.649185
WAPE     36.058059
Bias     -0.181577
fold      5.500000
dtype: float64

CV STD
MAE      0.124821
MSE      1.879313
MedAE    0.152632
R2       0.052675
MAPE     1.351498
SMAPE    0.705554
WAPE     0.759268
Bias     0.297232
fold     3.027650
dtype: float64


In [18]:
def bin_analysis(y_true, y_pred):
    dfb = pd.DataFrame({
        "true": y_true,
        "pred": y_pred
    })

    bins = pd.cut(
        dfb["true"],
        bins=[4.98, 9.99, 14.99, 19.99, 24.99, 29.99],
        labels=["4.99-9.99", "10-14.99", "15-19.99", "20-24.99", "25-29.99"]
    )

    dfb["bin"] = bins

    return dfb.groupby("bin").apply(lambda x: pd.Series({
        "count": len(x),
        "MAE": np.mean(np.abs(x["pred"] - x["true"])),
        "Bias": np.mean(x["pred"] - x["true"]),
        "MAPE": np.mean(np.abs((x["true"] - x["pred"]) / x["true"])) * 100
    }))

In [19]:
print(bin_analysis(oof_true, oof_pred))

             count        MAE       Bias       MAPE
bin                                                
4.99-9.99  17244.0   3.189648   2.810549  52.649695
10-14.99    5114.0   3.089185  -1.959876  22.293228
15-19.99    3804.0   5.896021  -5.645402  30.418536
20-24.99     957.0   9.661510  -9.657612  38.948483
25-29.99     931.0  13.770217 -13.768085  46.042326


C:\Users\alex\AppData\Local\Temp\ipykernel_10972\1811907578.py:15: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  return dfb.groupby("bin").apply(lambda x: pd.Series({
C:\Users\alex\AppData\Local\Temp\ipykernel_10972\1811907578.py:15: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  return dfb.groupby("bin").apply(lambda x: pd.Series({


In [20]:

def worst_predictions_table(y_true, y_pred, X=None, n=50):
    """
    Возвращает таблицу самых больших ошибок.
    
    y_true, y_pred: array-like
    X: optional features (DataFrame) для контекста
    n: сколько худших примеров вернуть
    """

    y_true = np.array(y_true)
    y_pred = np.array(y_pred)

    residual = y_pred - y_true
    abs_error = np.abs(residual)
    rel_error = residual / np.maximum(y_true, 1e-8)  # защита от 0

    df_out = pd.DataFrame({
        "y_true": y_true,
        "y_pred": y_pred,
        "error": residual,
        "abs_error": abs_error,
        "rel_error": rel_error
    })

    # добавляем фичи, если есть
    if X is not None:
        X_reset = X.reset_index(drop=True)
        df_out = pd.concat([df_out, X_reset], axis=1)

    # сортировка по абсолютной ошибке
    df_out = df_out.sort_values("rel_error", ascending=False)

    return df_out.head(n)

In [21]:
def worst_predictions_log_table(y_true, y_pred, X=None, n=50):
    y_true = np.array(y_true)
    y_pred = np.array(y_pred)

    log_residual = np.log1p(y_pred) - np.log1p(y_true)
    abs_log_error = np.abs(log_residual)

    df_out = pd.DataFrame({
        "y_true": y_true,
        "y_pred": y_pred,
        "log_residual": log_residual,
        "abs_log_error": abs_log_error
    })

    if X is not None:
        X_reset = X.reset_index(drop=True)
        df_out = pd.concat([df_out, X_reset], axis=1)

    df_out = df_out.sort_values("abs_log_error", ascending=False)

    return df_out.head(n)

In [22]:
oof_df = worst_predictions_table(
    oof_true,
    oof_pred,
    X=df,   # или None
    n=100
)

print(oof_df)

       y_true     y_pred      error  abs_error  rel_error   app_id  \
12350    4.99  22.112148  17.122148  17.122148   3.431292  1117330   
10649    4.99  21.564750  16.574750  16.574750   3.321593   780350   
19788    4.99  21.530240  16.540240  16.540240   3.314677  1361690   
14512    4.99  21.360121  16.370121  16.370121   3.280585   651670   
21322    4.99  20.970095  15.980095  15.980095   3.202424  2094250   
...       ...        ...        ...        ...        ...      ...   
18936    5.99  18.823903  12.833903  12.833903   2.142555  1147550   
22987    4.99  15.680312  10.690312  10.690312   2.142347  1707650   
27225    4.99  15.668381  10.678381  10.678381   2.139956  2249440   
9578     5.59  17.551080  11.961080  11.961080   2.139728   820040   
14526    4.99  15.657433  10.667433  10.667433   2.137762  1052990   

                                           name             developers  \
12350                     Bouncy Bob: Episode 2           MadGamesmith   
10649      

In [23]:
print("TOP 10 WORST ERRORS")
print(
    oof_df[["y_true", "y_pred", "error", "abs_error", "rel_error"]].head(10)
)

TOP 10 WORST ERRORS
       y_true     y_pred      error  abs_error  rel_error
12350    4.99  22.112148  17.122148  17.122148   3.431292
10649    4.99  21.564750  16.574750  16.574750   3.321593
19788    4.99  21.530240  16.540240  16.540240   3.314677
14512    4.99  21.360121  16.370121  16.370121   3.280585
21322    4.99  20.970095  15.980095  15.980095   3.202424
27789    5.39  22.302381  16.912381  16.912381   3.137733
13603    4.99  20.409001  15.419001  15.419001   3.089980
24741    4.99  20.286812  15.296812  15.296812   3.065493
16512    4.99  19.333147  14.343147  14.343147   2.874378
8782     4.99  19.278367  14.288367  14.288367   2.863400


In [24]:
oof_df["price_bin"] = pd.cut(
    oof_df["y_true"],
    bins=[0, 10, 20, 30, 40, 50, 1000],
    include_lowest=True
)

print(
    oof_df.groupby("price_bin")[["abs_error", "error"]].mean()
)

                abs_error      error
price_bin                           
(-0.001, 10.0]  12.763902  12.763902
(10.0, 20.0]          NaN        NaN
(20.0, 30.0]          NaN        NaN
(30.0, 40.0]          NaN        NaN
(40.0, 50.0]          NaN        NaN
(50.0, 1000.0]        NaN        NaN


C:\Users\alex\AppData\Local\Temp\ipykernel_10972\3155619407.py:8: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  oof_df.groupby("price_bin")[["abs_error", "error"]].mean()


In [25]:
oof_df_log = worst_predictions_log_table(
    oof_true,
    oof_pred,
    X=df,
    n=100
)

print(oof_df_log)

       y_true     y_pred  log_residual  abs_log_error   app_id  \
12350    4.99  22.112148      1.350267       1.350267  1117330   
10649    4.99  21.564750      1.326298       1.326298   780350   
19788    4.99  21.530240      1.324767       1.324767  1361690   
14512    4.99  21.360121      1.317188       1.317188   651670   
21322    4.99  20.970095      1.299591       1.299591  2094250   
...       ...        ...           ...            ...      ...   
5576     5.99  20.169380      1.108075       1.108075   520850   
15929    4.99  17.131337      1.107550       1.107550   502470   
19963    4.99  17.063801      1.103819       1.103819  1082450   
16966   29.99   9.286950     -1.102788       1.102788  1589380   
17589   29.99   9.298035     -1.101711       1.101711  1599920   

                                   name                        developers  \
12350             Bouncy Bob: Episode 2                      MadGamesmith   
10649                     Unruly Heroes              